# 트랜스포머 전체 구현 (처음부터) - 완전한 어텐션 전용 트랜스포머

- Tutorial ID: `tut-7-1`
- Tutorial: 트랜스포머 전체 구현 (처음부터)
- Section ID: `s7-1-1`
- Section: 완전한 어텐션 전용 트랜스포머


In [ ]:
# ============================================================
# 코드 읽는 법 — 완전한 어텐션 전용 트랜스포머
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 60)
print("어텐션 전용 트랜스포머: 완전 구현 (함수형)")
print("=" * 60)

def softmax(x, axis=-1):
        x = x - np.max(x, axis=axis, keepdims=True)
    return np.exp(x) / np.sum(np.exp(x), axis=axis, keepdims=True)

def layer_norm(x, eps=1e-5):
    mean = np.mean(x, axis=-1, keepdims=True)
    var = np.var(x, axis=-1, keepdims=True)
    return (x - mean) / np.sqrt(var + eps)

In [ ]:
# 어텐션 전용 트랜스포머 (함수 + 딕셔너리)
# class를 사용하지 않고 순수 함수만으로 구현

In [ ]:
def sinusoidal_encoding(max_len, d_model):
    pe = np.zeros((max_len, d_model))
    pos = np.arange(max_len)[:, np.newaxis]
    div = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(pos * div)
    pe[:, 1::2] = np.cos(pos * div[:d_model//2])
    return pe

def create_model(vocab_size, d_model, n_heads, n_layers, max_seq_len):
        np.random.seed(42)
    scale = 0.02
        d_head = d_model // n_heads
        
    model = {
        'W_E': np.random.randn(vocab_size, d_model) * scale,
        'W_U': np.random.randn(d_model, vocab_size) * scale,
        'b_U': np.zeros(vocab_size),
        'pos_embed': sinusoidal_encoding(max_seq_len, d_model),
        'n_heads': n_heads, 'd_head': d_head,
        'layers': [],
    }
        for _ in range(n_layers):
        model['layers'].append({
            'W_Q': [np.random.randn(d_model, d_head) * scale for _ in range(n_heads)],
            'W_K': [np.random.randn(d_model, d_head) * scale for _ in range(n_heads)],
            'W_V': [np.random.randn(d_model, d_head) * scale for _ in range(n_heads)],
            'W_O': [np.random.randn(d_head, d_model) * scale for _ in range(n_heads)],
            'ln_w': np.ones(d_model), 'ln_b': np.zeros(d_model),
        })
    return model

def attention_head_fn(x, W_Q, W_K, W_V, W_O, d_head, mask):
    """단일 어텐션 헤드"""
        Q = x @ W_Q
        K = x @ W_K
        V = x @ W_V
        scores = Q @ K.T / np.sqrt(d_head) + mask
        attn_weights = softmax(scores)
    return (attn_weights @ V) @ W_O, attn_weights

def forward_pass(model, token_ids, return_attentions=False):
    """전방 패스"""
        seq_len = len(token_ids)
        mask = np.triu(np.full((seq_len, seq_len), -1e9), k=1)
        x = model['W_E'][token_ids] + model['pos_embed'][:seq_len]
    all_attentions = []
    
        for layer in model['layers']:
        x_norm = layer_norm(x) * layer['ln_w'] + layer['ln_b']
                layer_out = np.zeros_like(x)
        layer_attns = []
                for h in range(model['n_heads']):
            head_out, attn_w = attention_head_fn(
                x_norm, layer['W_Q'][h], layer['W_K'][h],
                layer['W_V'][h], layer['W_O'][h],
                model['d_head'], mask)
            layer_out += head_out
            layer_attns.append(attn_w)
                x = x + layer_out
        all_attentions.append(layer_attns)
    
        logits = layer_norm(x) @ model['W_U'] + model['b_U']
    return (logits, all_attentions) if return_attentions else logits

In [ ]:
# 모델 테스트

In [ ]:
vocab = ['<BOS>', 'A', 'B', 'C', 'D', '<EOS>']
vocab_size = len(vocab)

model = create_model(vocab_size, d_model=16, n_heads=2, n_layers=2, max_seq_len=20)

print(f"모델 파라미터:")
print(f"  vocab_size={vocab_size}, d_model=16, n_heads=2, n_layers=2")

def count_params(m):
    total = m['W_E'].size + m['W_U'].size
        for layer in m['layers']:
                for h in range(m['n_heads']):
            total += sum(layer[k][h].size for k in ['W_Q','W_K','W_V','W_O'])
        total += layer['ln_w'].size + layer['ln_b'].size
    return total

n_params = count_params(model)
print(f"  총 파라미터 수: {n_params:,}")

# 전방 패스 테스트
sequence = [0, 1, 2, 3, 1, 2]  # BOS A B C A B
print(f"
입력 시퀀스: {[vocab[t] for t in sequence]}")

logits, attentions = forward_pass(model, sequence, return_attentions=True)
probs = softmax(logits)

print(f"
각 위치의 다음 토큰 예측 (상위 2개):")
for pos, (tok_id, pos_probs) in enumerate(zip(sequence, probs)):
    top2 = np.argsort(pos_probs)[-2:][::-1]
    preds = [(vocab[i], f"{pos_probs[i]:.3f}") for i in top2]
    print(f"  위치 {pos} ({vocab[tok_id]:4s}) → {preds}")

# 어텐션 패턴 시각화
print("
레이어 0, 헤드 0 어텐션 패턴:")
print("     " + " ".join(f"{vocab[t]:4s}" for t in sequence))
A = attentions[0][0]
for i, t in enumerate(sequence):
    row = " ".join(f"{A[i,j]:.2f}" for j in range(len(sequence)))
    print(f"  {vocab[t]}: {row}")

print("
✅ 어텐션 전용 트랜스포머 구현 완료!")
print("이 구현이 논문의 핵심 수식을 모두 포함합니다.")